# Model Training and Evaluation

This notebook is the **third stage** in the machine learning pipeline and builds 
directly upon the preprocessed data from the previous notebook. Our goal here is 
to **train, tune, and evaluate machine learning models** to predict whether a 
diabetic patient will be readmitted within 30 days of discharge.

---

### Notebook Objective

In this notebook, we aim to:
- Load the preprocessed train/validation/test sets
- Establish a baseline model to benchmark against
- Train and tune multiple machine learning models
- Evaluate models using appropriate metrics for imbalanced data
- Select the best performing model for explainability analysis

---

### Why This Matters

Model selection and evaluation are critical steps in any ML pipeline. A poorly 
evaluated model can appear to perform well while learning nothing useful — 
especially on imbalanced datasets like this one where a naive model can achieve 
91% accuracy by always predicting the majority class. Rigorous evaluation ensures 
our model is genuinely learning signal, not exploiting class imbalance.

---

### Evaluation Strategy

Since the positive class (readmitted within 30 days) represents only ~9% of the 
data, accuracy is not a meaningful metric. We will prioritize:
- **F1-score** — balances precision and recall
- **ROC-AUC** — measures discriminative ability across thresholds
- **Precision/Recall** — to understand the tradeoff between false positives and false negatives

---

### Next Steps in the Workflow

The best model selected here will be used in the following notebook:
- `04_model_explainability.ipynb`

## Section 1 - Imports and Data Loading

In [1]:
# Importing the libaries needed 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random


# Setting the plot style
sns.set(style='whitegrid')

# Setting a random state for reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [2]:
import os
import sys
sys.path.append(os.path.abspath('..'))

In [3]:
# Importing the custom functions, reloading the module to ensure the latest version is used
import importlib
import utils.functions as f

importlib.reload(f)


<module 'utils.functions' from 'c:\\Users\\Elias\\Documents\\Diabetes-readmission-ml-pipeline-main\\Diabetes-readmission-ml-pipeline-main\\utils\\functions.py'>

In [4]:
# Loading the processed data, splitted into training, validation, and test sets
# Training
X_train = pd.read_csv('../data/processed/X_train.csv', index_col=0)
y_train = pd.read_csv('../data/processed/y_train.csv', index_col=0).squeeze()

# Validation
X_val = pd.read_csv('../data/processed/X_val.csv', index_col=0)
y_val = pd.read_csv('../data/processed/y_val.csv', index_col=0).squeeze()

# Test
X_test = pd.read_csv('../data/processed/X_test.csv', index_col=0)
y_test = pd.read_csv('../data/processed/y_test.csv', index_col=0).squeeze()

In [5]:
# Verification
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"NaN values: {X_train.isnull().sum().sum()}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True)}")

X_train: (41955, 38), y_train: (41955,)
X_val: (13993, 38), y_val: (13993,)
X_test: (13981, 38), y_test: (13981,)
NaN values: 0
Target distribution:
readmitted
0    0.910237
1    0.089763
Name: proportion, dtype: float64


## Section 2 - Baseline Model
For this section, we are creating a baseline model, or dummy classifier, that will serveas a benchmark for the rest of the models. 

### 2.1 - Model Training

In [6]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_val)
y_prob = baseline.predict_proba(X_val)[:, 1]

baseline_metrics = f.evaluate_model(y_val, y_pred, y_prob)

print("Baseline Model Performance:")
print(baseline_metrics)

Baseline Model Performance:
{'accuracy': 0.9104552276138069, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'roc_auc': 0.5}


### 2.2 - Baseline Results

The dummy classifier achieves 91% accuracy by predicting the majority class (not 
readmitted) for every patient. This confirms that accuracy is a misleading metric 
on this imbalanced dataset, a model that never identifies a single readmission 
case scores 91%.

Our real models must demonstrate meaningful F1-score and ROC-AUC > 0.5 to be 
considered useful. These are the metrics we will prioritize going forward.

## Section 3 - Logistic Regression